# Week 01 — BBO capstone driver

Round 1 of 13. Eight opaque functions of dimension 2 to 8; I submit a vector in roughly [0.001, 1] and the portal returns one scalar. Maximise each.

**This round has no data at all.** With zero returned observations there is no posterior to compute Expected Improvement over, so a GP would be proposing from its prior and nothing else. Rather than dress that up, I take one arbitrary vector and slice it to each function's dimension — an honest cold-start probe that costs one round and establishes a scale for every output.

The trade-off I am accepting: eight correlated queries instead of eight independent ones. A Sobol or Latin hypercube design would spread the same budget far better. Noted now, acted on far too late.

In [ ]:
%matplotlib inline
import os, sys, warnings
warnings.filterwarnings("ignore")
# Walk up until bbo.py is found, so the notebook runs from anywhere in the repo.
_root = os.getcwd()
while not os.path.exists(os.path.join(_root, "bbo.py")) and os.path.dirname(_root) != _root:
    _root = os.path.dirname(_root)
os.chdir(_root); sys.path.insert(0, _root)
import numpy as np
import pandas as pd
import bbo

WEEK = 1
PRIOR = WEEK - 1          # data state this round was proposed from
SEED = 1
OUTDIR = f"outputs/week{WEEK:02d}"; os.makedirs(OUTDIR, exist_ok=True)

# What each function is getting this round, and why.
PLAN = {
    1: 'cold start — slice of one shared vector',
    2: 'cold start — slice of one shared vector',
    3: 'cold start — slice of one shared vector',
    4: 'cold start — slice of one shared vector',
    5: 'cold start — slice of one shared vector',
    6: 'cold start — slice of one shared vector',
    7: 'cold start — slice of one shared vector',
    8: 'cold start — slice of one shared vector',
}
pd.DataFrame([dict(func=f"F{f}", d=bbo.DIMS[f], move=PLAN[f]) for f in bbo.FUNC_IDS])


## 1. Data — nothing yet

No queries have been returned. There is no surrogate to fit and no incumbent to anchor on, so this round is a pure cold-start probe.


In [ ]:
print("no returned observations yet — cold start")
pd.DataFrame([dict(func=f"F{f}", d=bbo.DIMS[f], n_data=0) for f in bbo.FUNC_IDS])


## 2. Proposals — one vector, sliced

`BASE` is the 8-D seed; function *i* takes its first `DIMS[i]` coordinates.

In [ ]:
BASE = np.array([0.156843, 0.587493, 0.741585, 0.985632,
                 0.148524, 0.179638, 0.541596, 0.396585])

proposals = {fid: BASE[:bbo.DIMS[fid]].copy() for fid in bbo.FUNC_IDS}

pd.DataFrame([dict(func=f"F{fid}", d=bbo.DIMS[fid],
                   submission=bbo.submission(proposals[fid]))
              for fid in bbo.FUNC_IDS])


## 3. Visualise

Best-so-far trajectory per function, truncated to the data available this round.


In [ ]:
import matplotlib
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 4, figsize=(15, 6))
for ax, fid in zip(axes.ravel(), bbo.FUNC_IDS):
    try:
        _, y, rounds = bbo.load(fid, up_to=PRIOR)
    except ValueError:
        ax.set_title(f"F{fid}: no data"); continue
    ax.plot(rounds, y, "o", ms=4, alpha=.55)
    ax.plot(rounds, np.maximum.accumulate(y), "-", lw=2)
    ax.set_title(f"F{fid} (d={bbo.DIMS[fid]})", fontsize=9)
    ax.tick_params(labelsize=7); ax.set_xlabel("round", fontsize=8)
fig.suptitle(f"Best so far through round {PRIOR}", fontsize=11)
fig.tight_layout(); fig.savefig(f"{OUTDIR}/trajectories.png", dpi=140)
plt.show()


## 4. Submission strings


In [ ]:
# Portal format: six decimals, dash-separated, one line per function, no labels.
for fid in bbo.FUNC_IDS:
    print(bbo.submission(proposals[fid]))


## 5. After the portal returns each y

Returns recorded below and folded into `bbo.HISTORY` so the next round sees them.


In [ ]:
# Week 1 portal returns - already folded into bbo.HISTORY.
# returned_y = {
#     1: -5.768712955574306e-69,
#     2: 0.008511217799492411,
#     3: -0.12884849611277463,
#     4: -24.426727116824043,
#     5: 566.341996544051,
#     6: -0.8632458340875497,
#     7: 0.08028539779449592,
#     8: 7.429392424567499,
# }
#
# F5 returned 566.34 from this arbitrary vector. That is the highest value any
# function returned in the entire campaign and it was never beaten. Worth sitting
# with: a cold-start probe out-performed twelve subsequent rounds of optimisation.
#
# for fid, y in returned_y.items():
#     bbo.append_result(fid, proposals[fid], y, rnd=WEEK)
